In [1]:
import tensorflow as tf
import numpy as np
import os
print(os.getcwd())
tfrecordpath = "../Data/tfrecords/"


2025-01-31 15:49:42.788021: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1738334982.805494   18058 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1738334982.809001   18058 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-31 15:49:42.819192: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


/mnt/c/Users/alexs/Desktop/levbot/Training


### Load in data
#### Define schema


In [36]:
def decode(record_bytes):
    # Function for parsing each record in the tf files
    example = tf.io.parse_single_example(
        # Data
        record_bytes,

        # Schema
        {
        'Timeframe': tf.io.FixedLenFeature([], tf.string),
        'timestamp': tf.io.RaggedFeature(dtype=tf.int64),
        'open': tf.io.RaggedFeature(dtype=tf.float32),
        }
        )

    return example

In [37]:
path = tfrecordpath + "BTCUSD_PERP/test.tfrecord"

dataset = tf.data.TFRecordDataset(path,  num_parallel_reads = tf.data.AUTOTUNE)
dataset = dataset.map(decode, num_parallel_calls = tf.data.AUTOTUNE)



In [44]:
for data in dataset:
    print(data["timestamp"][0])

tf.Tensor(1597129799, shape=(), dtype=int64)


In [47]:
import collections
class WindowSlider:

    features = ("Open", "High", "Low", "Close", "Volume")

    def __init__(self, windowsize, lookforward, baseTimeframeDataset, otherdatasets):

        # Set up variables
        self.windowsize = windowsize
        self.lookforward = lookforward

        self.base = baseTimeframeDataset
        """Tensor dict"""
        self.others = otherdatasets
        """List of tensor dicts"""


        # The +1 is handled by the fact that the index is 1 less than the total amount
        self.baseindex = self.windowsize + self.lookforward
        """Index where the most forward portion of the buffer is"""
        self.otherindexes = [self.windowsize] * len(self.others)
        """List of indexes where the most forward portion of the buffer is"""

    def stepToPresent(self, buffer, timestamp = None):

        # Set current time
        if timestamp is None:
            ct = self.base["timestamp"][-(self.lookforward + 1)]
        else:
            ct = timestamp

        for i, ds in enumerate(self.others):
            # Check whether margin is before the current time
            while ct > ds["timestamp"][self.otherindexes[i]]:
                # We are not there yet, increase the index
                self.otherindexes[i] += 1

    def getLatestTime(self):
        """
        Returns the latest timestamp of the other timeframes
        """
        time = self.base["timestamp"][self.baseindex]

        for i, index in enumerate(self.otherindexes):
            latest = self.others[i]["timestamp"][index].value -1 # Remove one seconds to prevent fuckups wrt exact timing
            if latest > time:
                time = latest

        return time

    def __iter__(self):
        """
        Initializes the window slider from beginning
        """

        # Call iter on each dataset
        self.base = self.base.__iter__().__next__()
        for i, o in enumerate(self.others):
            self.others[i] = o.__iter__().__next__()

        # Reset indexes
        # The +1 is handled by the fact that the index is 1 less than the total amount
        self.baseindex = self.windowsize + self.lookforward
        self.otherindexes = [self.windowsize] * len(self.others)

        # Get latest time and step everyone to it
        latest = self.getLatestTime()
        self.stepToPresent(latest)

        return self

    def __next__(self):
        response = {}
        return self.base["timestamp"][self.baseindex - self.lookforward]



In [48]:
WS = WindowSlider(30, 10, dataset, [])

In [49]:
for time in WS:
    print(time)
    break

tf.Tensor(1597138799, shape=(), dtype=int64)


In [51]:
print(WS.baseindex)

40
